## Idea
1. Store alll the evidence into index and train the retriever by label evidence to get the right evidence for testing set
2. Trsin the transformer model to compare claim and evidence to predict the class

For labelled dataset we compare the claim with the given label evidence; for unlabeled dataset(i.e. testing set) we compare claims with our retrieved evidence to predict the class

## Imports and configuration

In [ ]:
# !pip install torch transformers scikit-learn rank-bm25 tqdm numpy faiss-cpu

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Change the current working directory to the NLP folder in Google Drive
os.chdir('/content/drive/MyDrive/NLP/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import json
import re
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

LABEL2ID = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


## Read files

In [11]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

EVIDENCE_PATH = DATA_DIR / "evidence.json"
TRAIN_PATH = DATA_DIR / "train-claims.json"
DEV_PATH = DATA_DIR / "dev-claims.json"
TEST_PATH = DATA_DIR / "test-claims-unlabelled.json"

MODEL_NAME = "distilroberta-base"
MODEL_DIR = OUTPUT_DIR / "verifier_model"


TEST_PRED_PATH = OUTPUT_DIR / "test_predictions.json"

## Utility functions

load/save file, get token/items, normalise text, concat evidence

In [12]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def normalise_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%°]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_tokenise(text: str) -> List[str]:
    return normalise_text(text).split()


def get_claim_items(claims_json: Dict) -> List[Tuple[str, Dict]]:
    return list(claims_json.items())


def concatenate_evidence(
    evidence_ids: List[str],
    evidence_corpus: Dict[str, str],
    max_evidence: int = 5
) -> str:
    selected_ids = evidence_ids[:max_evidence]
    evidence_texts = []

    for eid in selected_ids:
        if eid in evidence_corpus:
            evidence_texts.append(evidence_corpus[eid])

    if len(evidence_texts) == 0:
        return "No relevant evidence found."

    return " ".join(evidence_texts)

## Load data

In [13]:
evidence_corpus = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_PATH)
dev_claims = load_json(DEV_PATH)

print("Number of evidence passages:", len(evidence_corpus))
print("Number of train claims:", len(train_claims))
print("Number of dev claims:", len(dev_claims))

Number of evidence passages: 1208827
Number of train claims: 1228
Number of dev claims: 154


# Retriever

In [21]:
class BM25Retriever:
    def __init__(self, evidence_corpus: Dict[str, str]):
        self.evidence_corpus = evidence_corpus
        self.evidence_ids = list(evidence_corpus.keys())
        self.evidence_texts = [evidence_corpus[eid] for eid in self.evidence_ids]

        print("Building BM25 index...")
        tokenised_corpus = [simple_tokenise(text) for text in tqdm(self.evidence_texts)]
        self.bm25 = BM25Okapi(tokenised_corpus)

    def retrieve(self, claim_text: str, top_k: int = 5) -> List[str]:
        query_tokens = simple_tokenise(claim_text)
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [self.evidence_ids[i] for i in top_indices]

    def evaluate_recall_at_k(self, claims_json: Dict, k: int = 5) -> float:
        total = 0
        hit = 0

        for claim_id, instance in tqdm(get_claim_items(claims_json)):
            claim_text = instance["claim_text"]
            gold_evidence = set(instance.get("evidences", []))

            if len(gold_evidence) == 0:
                continue

            retrieved = set(self.retrieve(claim_text, top_k=k))

            if len(gold_evidence.intersection(retrieved)) > 0:
                hit += 1

            total += 1

        return hit / total if total > 0 else 0.0

##  Build and evaluate retriever

In [ ]:
retriever = BM25Retriever(evidence_corpus)

for k in [1, 3, 5, 10]:
    recall = retriever.evaluate_recall_at_k(dev_claims, k=k)
    print(f"Dev Retrieval Recall@{k}: {recall:.4f}")

Building BM25 index...


 60%|█████▉    | 92/154 [18:16<12:18, 11.92s/it]


KeyboardInterrupt: 

# Read Retriever files

In [16]:
import json
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer


class LoadedFAISSRetriever:
    def __init__(
        self,
        evidence_corpus,
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        cache_dir="faiss_cache",
        device=None
    ):
        self.evidence_corpus = evidence_corpus
        self.model_name = model_name
        self.cache_dir = Path(cache_dir)

        safe_model_name = model_name.replace("/", "_")
        self.index_path = self.cache_dir / f"{safe_model_name}.faiss"
        self.ids_path = self.cache_dir / f"{safe_model_name}_evidence_ids.json"

        if not self.index_path.exists():
            raise FileNotFoundError(f"Cannot find FAISS index: {self.index_path}")

        if not self.ids_path.exists():
            raise FileNotFoundError(f"Cannot find evidence IDs: {self.ids_path}")

        print("Loading SentenceTransformer model...")
        self.model = SentenceTransformer(model_name, device=device)

        print("Loading FAISS index...")
        self.index = faiss.read_index(str(self.index_path))

        print("Loading evidence ID mapping...")
        with open(self.ids_path, "r", encoding="utf-8") as f:
            self.evidence_ids = json.load(f)

        if len(self.evidence_ids) != self.index.ntotal:
            raise ValueError(
                f"Mismatch: {len(self.evidence_ids)} evidence IDs but "
                f"{self.index.ntotal} FAISS vectors."
            )

        print("FAISS retriever loaded successfully.")
        print("Number of indexed evidence passages:", self.index.ntotal)

    def retrieve(self, claim_text, top_k=5):
        claim_embedding = self.model.encode(
            [claim_text],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = self.index.search(claim_embedding, top_k)

        retrieved_ids = [self.evidence_ids[i] for i in indices[0]]
        return retrieved_ids

In [18]:
faiss_retriever = LoadedFAISSRetriever(
    evidence_corpus=evidence_corpus,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_dir="/content/drive/MyDrive/NLP/outputs/faiss_cache",
    device=device
)

Loading SentenceTransformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading FAISS index...
Loading evidence ID mapping...
FAISS retriever loaded successfully.
Number of indexed evidence passages: 1208827


# Transformer Verifier

## Get dataset

In [22]:
class ClaimEvidenceDataset(Dataset):
    def __init__(
        self,
        claims_json: Dict,
        evidence_corpus: Dict[str, str],
        tokenizer,
        max_length: int = 512,
        max_evidence: int = 5,
        use_gold_evidence: bool = True,
        retriever: BM25Retriever = None,
        retrieval_top_k: int = 5,
        is_test: bool = False
    ):
        self.items = get_claim_items(claims_json)
        self.evidence_corpus = evidence_corpus
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_evidence = max_evidence
        self.use_gold_evidence = use_gold_evidence
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        if self.use_gold_evidence and not self.is_test:
            evidence_ids = instance.get("evidences", [])
        else:
            if self.retriever is None:
                raise ValueError("Retriever is required when not using gold evidence.")
            evidence_ids = self.retriever.retrieve(claim_text, top_k=self.retrieval_top_k)

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence
        )

        encoded = self.tokenizer(
            claim_text,
            evidence_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "claim_id": claim_id,
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "evidence_ids": evidence_ids
        }

        if not self.is_test:
            label = instance["claim_label"]
            item["label"] = torch.tensor(LABEL2ID[label], dtype=torch.long)

        return item


def collate_fn(batch):
    input_ids = torch.stack([x["input_ids"] for x in batch])
    attention_mask = torch.stack([x["attention_mask"] for x in batch])
    claim_ids = [x["claim_id"] for x in batch]
    evidence_ids = [x["evidence_ids"] for x in batch]

    output = {
        "claim_ids": claim_ids,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": evidence_ids
    }

    if "label" in batch[0]:
        labels = torch.stack([x["label"] for x in batch])
        output["labels"] = labels

    return output

## Training and evaluation functions

In [23]:
def evaluate_verifier(model, dataloader, device: str):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    print(classification_report(
        all_labels,
        all_preds,
        target_names=[ID2LABEL[i] for i in range(4)]
    ))

    return acc, macro_f1


def train_verifier(
    train_claims: Dict,
    dev_claims: Dict,
    evidence_corpus: Dict[str, str],
    model_name: str = "distilroberta-base",
    output_dir: Path = Path("outputs/verifier_model"),
    epochs: int = 3,
    batch_size: int = 8,
    lr: float = 2e-5,
    max_length: int = 512,
    max_evidence: int = 5,
    device: str = "cpu"
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=4,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )
    model.to(device)

    train_dataset = ClaimEvidenceDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        tokenizer=tokenizer,
        max_length=max_length,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    dev_dataset = ClaimEvidenceDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        tokenizer=tokenizer,
        max_length=max_length,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    optimiser = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_macro_f1 = 0.0

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            optimiser.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimiser.step()
            scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Training loss: {avg_loss:.4f}")

        dev_acc, dev_macro_f1 = evaluate_verifier(model, dev_loader, device)
        print(f"Dev accuracy with gold evidence: {dev_acc:.4f}")
        print(f"Dev macro F1 with gold evidence: {dev_macro_f1:.4f}")

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            output_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
            print(f"Saved best model to {output_dir}")

    return model, tokenizer

## Train the Transformer verifier

This trains using **gold evidence** from the labelled training data.

In [24]:
# You can reduce epochs or batch size if training is slow.
model, tokenizer = train_verifier(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    model_name=MODEL_NAME,
    output_dir=MODEL_DIR,
    epochs=5,
    batch_size=8,
    lr=2e-5,
    max_length=512,
    max_evidence=5,
    device=device
)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Training loss: 0.5464


  0%|          | 0/20 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

       SUPPORTS       0.71      0.79      0.75        68
        REFUTES       0.55      0.59      0.57        27
NOT_ENOUGH_INFO       0.84      0.88      0.86        41
       DISPUTED       0.50      0.17      0.25        18

       accuracy                           0.71       154
      macro avg       0.65      0.61      0.61       154
   weighted avg       0.69      0.71      0.69       154

Dev accuracy with gold evidence: 0.7078
Dev macro F1 with gold evidence: 0.6071


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model to outputs/verifier_model


##  Full pipeline prediction function
This uses retrieved evidence instead of gold evidence. Use this for dev full-pipeline testing and test prediction.

In [25]:
def predict_claims(
    claims_json: Dict,
    evidence_corpus: Dict[str, str],
    retriever: BM25Retriever,
    model_path: Path,
    output_path: Path,
    retrieval_top_k: int = 10,
    max_evidence: int = 3,
    batch_size: int = 8,
    max_length: int = 512,
    is_test: bool = False,
    device: str = "cpu"
):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()

    dataset = ClaimEvidenceDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        tokenizer=tokenizer,
        max_length=max_length,
        max_evidence=max_evidence,
        use_gold_evidence=False,
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=is_test
    )

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_ids = torch.argmax(outputs.logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(batch["claim_ids"], pred_ids, batch["evidence_ids"]):
                predictions[claim_id] = {
                    "claim_label": ID2LABEL[pred_id],
                    "evidences": evidence_ids[:max_evidence]
                }

    save_json(predictions, output_path)
    print(f"Saved predictions to {output_path}")
    return predictions

## Generate dev predictions using retrieved evidence
This creates a prediction file that can be evaluated using the provided eval.py script.

note:

retrieval_top_k=10: retrieve 10 candidates internally.

max_evidence=3: only output the best 3 evidence IDs.

This often gives a better precision/recall balance than outputting all 10.

In [1]:
DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions_transformer_batch8.json"

dev_predictions = predict_claims(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model_path=MODEL_DIR,
    output_path=DEV_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=3,
    batch_size=8,
    max_length=512,
    is_test=False,
    device=device
)

NameError: name 'OUTPUT_DIR' is not defined

## evaulation by eval.py

In [ ]:
# Run the official evaluator on dev predictions.


!python eval.py --predictions outputs/dev_predictions_transformer_batch8.json --groundtruth data/dev-claims.json

Evidence Retrieval F-score (F)    = 0.08902288188002473
Claim Classification Accuracy (A) = 0.44155844155844154
Harmonic Mean of F and A          = 0.14817259202130112


## predict test claims

In [ ]:
test_claims = load_json(TEST_PATH)
TEST_PRED_PATH = OUTPUT_DIR / "test_predictions_transformer_batch8.json"
test_predictions = predict_claims(
    claims_json=test_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model_path=MODEL_DIR,
    output_path=TEST_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=3,
    batch_size=8,
    max_length=512,
    is_test=True,
    device=device
)

print("Test prediction file:", TEST_PRED_PATH)

100%|██████████| 20/20 [13:34<00:00, 40.73s/it]

Saved predictions to outputs\test_predictions.json
Test prediction file: outputs\test_predictions.json
